In [16]:
import arcpy
from arcpy import env
import os
import numpy as np
import arcgis
from arcgis import GIS
from arcgis.features import GeoAccessor
from arcgis.features import GeoSeriesAccessor
import pandas as pd

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"

# show all columns
pd.options.display.max_columns = None

# pd.DataFrame.spatial.from_featureclass(???)  
# df.spatial.to_featureclass(location=???,sanitize_columns=False)  

# gsa = arcgis.features.GeoSeriesAccessor(df['SHAPE'])  
# df['AREA'] = gsa.area  # KNOW YOUR UNITS

In [17]:
# # fill NA values in Spatially enabled dataframes (ignores SHAPE column)
# def fill_na_sedf(df_with_shape_column, fill_value=0):
#     if 'SHAPE' in list(df_with_shape_column.columns):
#         df = df_with_shape_column.copy()
#         shape_column = df['SHAPE'].copy()
#         del df['SHAPE']
#         return df.fillna(fill_value).merge(shape_column,left_index=True, right_index=True, how='inner')
#     else:
#         raise Exception("Dataframe does not include 'SHAPE' column")

In [18]:
if not os.path.exists('Outputs'):
    os.makedirs('Outputs')
    
outputs = ['.\\Outputs', "scratch.gdb", 'results.gdb']
gdb = os.path.join(outputs[0], outputs[1])
gdb2 = os.path.join(outputs[0], outputs[2])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])

if not arcpy.Exists(gdb2):
    arcpy.CreateFileGDB_management(outputs[0], outputs[2])

In [ ]:
taz_1000 = r".\Inputs\WFv1000_TAZ.shp"
sl_zones = r'..\_SaltLake\Inputs\Microzones_DRAFT.gdb\Microzones_DRAFT'
da_zones = r'..\_Davis\Inputs\Microzones_DRAFT.gdb\Microzones_DRAFT'
wbe_zones = r'..\_Weber_BoxElder\Inputs\Microzones_DRAFT.gdb\Microzones_DRAFT'
ut_zones = r'..\_Utah\Inputs\Microzones_DRAFT.gdb\Microzones_DRAFT'

In [20]:
sl_zones_lyr = arcpy.MakeFeatureLayer_management(sl_zones, 'sl_zones_lyr')
da_zones_lyr = arcpy.MakeFeatureLayer_management(da_zones, 'da_zones_lyr')
wbe_zones_lyr = arcpy.MakeFeatureLayer_management(wbe_zones, 'wbe_zones_lyr')
ut_zones_lyr = arcpy.MakeFeatureLayer_management(ut_zones, 'ut_zones_lyr')

In [21]:
q = '''Status = 1 AND CO_NAME IN ('SALT LAKE')'''
arcpy.SelectLayerByAttribute_management(sl_zones_lyr, 'NEW_SELECTION', q,)

q = '''Status = 1 AND CO_NAME IN ('DAVIS')'''
arcpy.SelectLayerByAttribute_management(da_zones_lyr, 'NEW_SELECTION', q,)

q = '''Status = 1 AND CO_NAME IN ('WEBER', 'BOX ELDER')'''
arcpy.SelectLayerByAttribute_management(wbe_zones_lyr, 'NEW_SELECTION', q,)

q = '''Status = 1 AND CO_NAME IN ('UTAH')'''
arcpy.SelectLayerByAttribute_management(ut_zones_lyr, 'NEW_SELECTION', q,)

<Result 'ut_zones_lyr'>

In [22]:
# merge all reviewed microzones into a single feature class
reviewed_microzones = arcpy.management.Merge([sl_zones_lyr, da_zones_lyr, wbe_zones_lyr, ut_zones_lyr], os.path.join(gdb, 'reviewed_microzones'),
                       add_source="ADD_SOURCE_INFO")

In [ ]:
arcpy.analysis.Identity(
    in_features=,
    identity_features=reviewed_microzones,
    out_feature_class=r"\\modelqueen\ModelQueen-D\ABM-INP-Microzones\Scripts\Default.gdb\TAZ_Boundary_Identity",
    join_attributes="ALL",
    cluster_tolerance=None,
    relationship="NO_RELATIONSHIPS"
)

In [23]:
# create zones from taz that don't have a reviewed microzone

# do stuff with identity
- polygons that are within a taz that has other reviewed tazs should be split up into tiny pieces and merged to adjacent zones
- the others can be left alone

# recut TAZ boundaries

# fill in missing areas

# create a temp maz id